In [1]:
import os
from pathlib import Path

os.getcwd()
mb_dir = Path(os.getcwd()).parent.parent.parent
os.chdir(mb_dir)
data_dir = str(mb_dir.parent / "data")
root_dir = str(mb_dir.parent)

In [2]:
import xarray as xr 
import numpy as np

from examples.paper_figures.data_utils import (
    get_model_dfs, get_plot_metrics, save_data,
    load_wyi, get_climatological_dfs, load_wyi,
    YEAR_RANGES, EXTENDED_YEARS, YEAR_RANGES_COM
)

from monsoonbench.metrics import (
    ClimatologyOnsetMetrics
)

from monsoonbench.visualization import create_model_comparison_table

c_metrics = ClimatologyOnsetMetrics()

# Figure 4 year ranges
YEAR_RANGES_COM = {
    "AIFS": np.arange(2004, 2022),
    "IFS": np.arange(2004, 2022),
    "FuXi": np.arange(2004, 2022),
    "Graphcast": np.arange(2004, 2022),
    "GenCast": np.arange(2019, 2022),
    "FuXi-S2S": np.arange(2004, 2022),
    "NGCM": np.arange(2004, 2022),
}

config = {
    "years": np.arange(2019, 2025),
    "extended_years": np.concatenate((np.arange(1965, 1979), np.arange(2019, 2025))),  # Extended period for analysis
    "common_years": np.arange(2004, 2022),
    "imd_folder": f"{data_dir}/imd_rainfall_data/4p0",  # Ground truth rainfall data (4x4 degrees)
    "thres_file": f"{data_dir}/imd_onset_threshold/mwset4x4.nc4",  # Threshold for the onset of the monsoon (4x4 degrees)
    "shpfile_path": f"{data_dir}/ind_map_shpfile/india_shapefile.shp",  # Shapefile of India
    "output_dir": f"{root_dir}/output",  # Directory to save data files
}

config2 = {
    "imd_folder": f"{data_dir}/imd_rainfall_data/4p0",  # Ground truth rainfall data (4x4 degrees)
    "thres_file": f"{data_dir}/imd_onset_threshold/mwset4x4.nc4",  # Threshold for the onset of the monsoon (4x4 degrees)
    "shpfile_path": f"{data_dir}/ind_map_shpfile/india_shapefile.shp",  # Shapefile of India
    "output_dir": f"{root_dir}/output",  # Directory to save data files
    "years": np.concatenate((np.arange(1965, 1979), np.arange(2019, 2025)))  # Extended period for analysis
}


config3 = {
    "imd_folder": f"{data_dir}/imd_rainfall_data/4p0",  # Ground truth rainfall data (4x4 degrees)
    "thres_file": f"{data_dir}/imd_onset_threshold/mwset4x4.nc4",  # Threshold for the onset of the monsoon (4x4 degrees)
    "shpfile_path": f"{data_dir}/ind_map_shpfile/india_shapefile.shp",  # Shapefile of India
    "output_dir": f"{root_dir}/output",  # Directory to save data files
    "years": np.arange(2004, 2022)  # Extended period for analysis
}

model_paths = {
    "IFS": f"{data_dir}/rainfall_4p0/IFS_S2S",
    "AIFS":  f"{data_dir}/rainfall_4p0/AIFS",
    "FuXi": f"{data_dir}/rainfall_4p0/FuXi",
    "Graphcast": f"{data_dir}/rainfall_4p0/GraphCast",
    "GenCast": f"{data_dir}/rainfall_4p0/GenCast",
    "FuXi-S2S": f"{data_dir}/rainfall_4p0/FuXi_S2S",
    "NGCM": f"{data_dir}/rainfall_4p0/NeuralGCM"
}


### Loading fig 3 ground truth data

In [4]:
import scipy.io as sio
weekly_file = f"{root_dir}/fig_data/5day_forecastwindow_cmz_2019_2024.mat"
data = sio.loadmat(weekly_file)
mae_cmz = data['mae_cmz']  # Shape should be (6, 8) for 6 time periods, 8 models
far_cmz = data['far_cmz']  # Shape should be (6, 8)
mr_cmz = data['mr_cmz']    # Shape should be (4, 8) for 4 weeks
std_er = data['std_er']    # Standard errors for MAE

print(data["model_str"])
mae_cmz
gt_mae = []
for r in far_cmz:
    gt_mae.append(r[0])

[[array(['clim'], dtype='<U4')]
 [array(['ifs'], dtype='<U3')]
 [array(['aifs'], dtype='<U4')]
 [array(['fuxi'], dtype='<U4')]
 [array(['graphcast'], dtype='<U9')]
 [array(['gencast'], dtype='<U7')]
 [array(['fuxis2s'], dtype='<U7')]
 [array(['ngcm51'], dtype='<U6')]]


Generating climatoligical window data

In [4]:
from examples.paper_figures.fig3_utils import (
    DEFAULT_WINDOW_BINS, get_clim_window_data,
    get_model_window_data
)

clim_data = get_clim_window_data(config)


🎨 Scientific plotting configuration loaded
   Default save format: png
   Contour levels: 100
   Colormap: bwr
Computing climatological onset reference...
Computing climatological onset from 124 years: 1901-2024
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1901.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1901-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1902.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1902-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1903.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1903-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1904.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd)

In [5]:
import pandas as pd
clim_window_data = {}

for i, (lower, upper) in enumerate(DEFAULT_WINDOW_BINS):
    clim_window_data[f"{lower}-{upper}"] = clim_data[i]


rep_df = create_model_comparison_table(clim_window_data)
print(rep_df.columns)
pd.DataFrame({
    "Correct FAR (clim)": gt_mae,
    "Reproduced FAR (clim)": rep_df["cmz_far_pct"].values
})

Index(['cmz_mae_mean_days', 'cmz_mae_se_days', 'cmz_far_pct', 'cmz_mr_pct',
       'overall_mae_mean_days', 'overall_far_pct', 'overall_mr_pct'],
      dtype='object')


,Correct FAR (clim),Reproduced FAR (clim)
0,5.005527,5.005527
1,7.959823,7.959823
2,10.514863,10.514863
3,12.040734,12.040734
4,14.469973,14.469973
5,16.131824,16.131824


In [ ]:
from monsoonbench.metrics import (
    ClimatologyOnsetMetrics,
    DeterministicOnsetMetrics,
    ProbabilisticOnsetMetrics,
)

import pandas as pd
windows = [1,6,11,16,21,26]



mp = {
    "FuXi": f"{data_dir}/rainfall_4p0/FuXi",
}

window_dfs = []

for start_date in windows:
    f3_df, f3_onsets = get_model_window_data(model_paths, YEAR_RANGES, config, lower_window=start_date)
    window_data = {}
    for model_name in model_paths.keys():
        plot_probabilistic_metrics = c_metrics.create_spatial_far_mr_mae(
            f3_df[model_name], f3_onsets[model_name]
        )
        window_data[model_name] = plot_probabilistic_metrics

    window_df = create_model_comparison_table(window_data)
    window_dfs.append(window_df)



Processing year 2019
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\2019.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (2019-06-02) as start date for onset detection
Processing 26 init times x 8 lats x 9 lons...
Using MOK (6/2 filter) for onset detection
Only processing forecasts initialized before observed onset dates
Requiring ≥50% of 11 members to have onset for ensemble onset
Processing init time 1/26: 2019-05-02
Processing init time 6/26: 2019-05-20


Exception ignored while calling weakref callback <function WeakValueDictionary.__init__.<locals>.remove at 0x0000020E869124B0>:
Traceback (most recent call last):
  File "C:\Users\cflor\AppData\Local\Python\pythoncore-3.14-64\Lib\weakref.py", line 105, in remove
    def remove(wr, selfref=ref(self), _atomic_removal=_remove_dead_weakref):
KeyboardInterrupt: 


Processing init time 11/26: 2019-06-06
Processing init time 16/26: 2019-06-24
Processing init time 21/26: 2019-07-11
Processing init time 26/26: 2019-07-29

Processing Summary:
Total potential initializations: 1872
Skipped (no observed onset): 962
Skipped (initialized after observed onset): 394
Valid initializations processed: 516
Ensemble onsets found (≥50% members): 64
Ensemble onset rate: 0.124
Note: Only onsets on or after 6/2 were counted due to MOK flag
Computing onset metrics with tolerance = 2 days
Verification window starts 1 days after initialization
Forecast window length: 6 days
Processing 35 unique grid points...
Processing grid point 1/35: lat=8.00, lon=76.00
Processing grid point 11/35: lat=20.00, lon=76.00
Processing grid point 21/35: lat=24.00, lon=92.00
Processing grid point 31/35: lat=32.00, lon=72.00
Year 2019 completed. Grid points processed: 35

Processing year 2020
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\2020.nc


In [13]:
window_dfs[3]

,cmz_mae_mean_days,cmz_mae_se_days,cmz_far_pct,cmz_mr_pct,overall_mae_mean_days,overall_far_pct,overall_mr_pct
model,,,,,,,
IFS,5.526720,1.112255,6.287367,20.333333,10.083046,9.299771,25.428571
AIFS,6.921252,0.691693,12.381626,18.888889,9.394426,15.798349,23.660562
FuXi,5.649173,0.601523,13.885213,34.722222,8.513600,14.137908,45.745527
Graphcast,8.466860,0.820215,24.138607,12.500000,10.083258,21.997944,19.695971
GenCast,5.451852,0.850280,6.121668,30.555556,8.984485,9.291935,29.139194
FuXi-S2S,5.991667,0.508333,8.051465,27.777778,8.273670,10.738984,31.632653
NGCM,6.148082,0.478828,6.826762,20.555556,9.603409,8.946788,31.393162


In [14]:
far = []
mae = []
std_er = []
mr = []
for i in range(len(window_dfs)):
    far_win = [rep_df.iloc[i]["cmz_far_pct"]]
    mae_win = [rep_df.iloc[i]["cmz_mae_mean_days"]]
    std_er_win = [rep_df.iloc[i]["cmz_mae_se_days"]]
    mr_win = [rep_df.iloc[i]["cmz_mr_pct"]]

    far_win += window_dfs[i]["cmz_far_pct"].values.tolist()
    mae_win += window_dfs[i]["cmz_mae_mean_days"].values.tolist()
    std_er_win += window_dfs[i]["cmz_mae_se_days"].values.tolist()
    mr_win += window_dfs[i]["cmz_mr_pct"].values.tolist()

    far.append(np.array(far_win))
    mae.append(np.array(mae_win))
    std_er.append(np.array(std_er_win))
    mr.append(np.array(mr_win))

out_dict = {
    "far_cmz": np.array(far),
    "mae_cmz": np.array(mae),
    "model_str": np.array(["clim", "ifs", "aifs", "fuxi", "graphcast",
                "gencast", "fuxis2s", "ngcm51"]),
    "std_er": np.array(std_er),
    "mr_cmz": np.array(mr)
}

In [17]:
out_dict["far_cmz"]

array([[ 5.00552672,  5.14806667,  6.14221866,  1.61055272,  8.92096044,
         4.60774198,  3.62910217,  5.85507257],
       [15.05474872,  8.78547899,  8.85290505,  4.26848899, 11.39773605,
         6.4457009 ,  5.40063474,  7.40127785],
       [24.6245394 ,  7.35800489, 12.77523487,  6.54565838, 13.09821372,
         4.49552206,  7.07672719,  4.00694431],
       [36.52676831,  6.28736664, 12.38162626, 13.88521342, 24.13860657,
         6.12166779,  8.05146487,  6.82676204],
       [48.69890291,  7.46808236, 17.49078923, 15.73044724, 27.4567295 ,
         6.71304634, 10.57814408, 10.33545782],
       [61.24031528, 10.74482817, 23.85337689, 20.23400135, 17.59355116,
         9.77286226, 14.10714286,  6.96542738]])

In [ ]:
from scipy.io import savemat


def save_data(mat_dict: dict[str, np.ndarray], output_dir: str, save_path: str) -> None:
    """Save data to a .mat file.

    Args:
        mat_dict: Dictionary of data to save.
        output_dir: Directory to save the data.
        save_path: Path to save the data.
    """
    out_path = f"{output_dir}/{save_path}.mat"
    savemat(out_path, mat_dict)
    print("Saved to:", out_path)
    return



Saved to: c:\Users\cflor\CSAssignments\Clinic3/output/5day_forecastwindow_cmz_2019_2024.mat.mat


In [6]:
import monsoonbench.spatial.regions as r 
cmz_lon, cmz_lat = r.get_cmz_polygon_coords(4)
grid_lats = np.unique(test_clim_forecast[2019]["lat"])
grid_lons = np.unique(test_clim_forecast[2019]["lon"])
r.points_inside_polygon(cmz_lon, cmz_lat, grid_lons, grid_lats)


(array([[False, False, False, False, False, False, False, False],
        [False, False, False, False, False, False, False, False],
        [False, False, False, False, False, False, False, False],
        [False, False,  True,  True,  True, False, False, False],
        [False,  True,  True,  True,  True, False, False, False],
        [False,  True,  True,  True, False, False, False, False],
        [False, False, False, False, False, False, False, False],
        [False, False, False, False, False, False, False, False]]),
 array([76., 80., 84., 72., 76., 80., 84., 72., 76., 80.]),
 array([20., 20., 20., 24., 24., 24., 24., 28., 28., 28.]))